In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
free  = torch.cuda.mem_get_info()[0] / 1e9
total = torch.cuda.mem_get_info()[1] / 1e9
print(f"GPU Memory: {free:.1f} GB free / {total:.1f} GB total")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ Seed set")

Device: cuda
GPU: Tesla T4
GPU Memory: 15.5 GB free / 15.6 GB total
✓ Seed set


In [2]:
# ============================================================
# CELL 1 — Download dataset from Kaggle
# ============================================================
from google.colab import files
import zipfile
import os

print("Please upload your kaggle.json file")
uploaded = files.upload()

dataset_slug = "tuanledinh/monuseg2018"
!kaggle datasets download -d {dataset_slug} -p /content/

zip_path = "/content/monuseg2018.zip"
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/MoNuSeg_Original")

print("Dataset extracted successfully!")


# ============================================================
# CELL 2 — Paths
# ============================================================
TRAIN_IMAGE_DIR = "/content/MoNuSeg_Original/kmms_training/kmms_training/images"
TRAIN_MASK_DIR  = "/content/MoNuSeg_Original/kmms_training/kmms_training/masks"
TEST_IMAGE_DIR  = "/content/MoNuSeg_Original/kmms_test/kmms_test/images"
TEST_MASK_DIR   = "/content/MoNuSeg_Original/kmms_test/kmms_test/masks"

TRAIN_PATCH_IMG  = "/content/MoNuSeg_Patches/train/images"
TRAIN_PATCH_MASK = "/content/MoNuSeg_Patches/train/masks"
TEST_PATCH_IMG   = "/content/MoNuSeg_Patches/test/images"
TEST_PATCH_MASK  = "/content/MoNuSeg_Patches/test/masks"

PATCH_SIZE     = 256
STRIDE         = 128   # 50% overlap
MIN_FOREGROUND = 0.02  # skip patches with <2% nucleus pixels


# ============================================================
# CELL 3 — Patch extraction (with mask cleanup + foreground filter)
# ============================================================
import cv2
import numpy as np
import shutil
from tqdm import tqdm

def fix_mask_filenames(mask_dir):
    """Remove accidental spaces from mask filenames (e.g. 'x .png' -> 'x.png')."""
    for f in os.listdir(mask_dir):
        old = os.path.join(mask_dir, f)
        new = os.path.join(mask_dir, f.replace(" ", ""))
        if old != new:
            os.rename(old, new)
    print(f"Mask filenames cleaned: {mask_dir}")


def extract_patches(image_dir, mask_dir, out_img_dir, out_mask_dir,
                     patch_size, stride, min_fg, split_name):
    """
    Extract overlapping patches from all images in image_dir.
    - Masks are binarized (0/255) before saving.
    - Patches with too little foreground (nucleus) are skipped.
    """
    for d in [out_img_dir, out_mask_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d)

    count, skipped = 0, 0
    image_files = sorted(os.listdir(image_dir))

    for img_name in tqdm(image_files, desc=f"Patching {split_name}"):
        img_path  = os.path.join(image_dir, img_name)
        mask_name = img_name.replace(".tif", ".png").replace(" ", "")
        mask_path = os.path.join(mask_dir, mask_name)

        image = cv2.imread(img_path)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            print(f"  Missing image: {img_path}")
            continue
        if mask is None:
            print(f"  Missing mask : {mask_path}")
            continue

        H, W = image.shape[:2]
        if mask.shape[:2] != (H, W):
            mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

        # Binarize: avoids stray pixel values (1-254) silently corrupting BCE loss
        mask = (mask > 127).astype(np.uint8) * 255

        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):
                img_patch  = image[y:y+patch_size, x:x+patch_size]
                mask_patch = mask[y:y+patch_size,  x:x+patch_size]

                if img_patch.shape != (patch_size, patch_size, 3):
                    continue
                if mask_patch.shape != (patch_size, patch_size):
                    continue

                fg_ratio = np.sum(mask_patch > 0) / (patch_size * patch_size)
                if fg_ratio < min_fg:
                    skipped += 1
                    continue

                cv2.imwrite(os.path.join(out_img_dir,  f"{count:05d}.png"), img_patch)
                cv2.imwrite(os.path.join(out_mask_dir, f"{count:05d}.png"), mask_patch)
                count += 1

    print(f"{split_name}: {count} patches saved | {skipped} background patches skipped")
    return count


fix_mask_filenames(TRAIN_MASK_DIR)
fix_mask_filenames(TEST_MASK_DIR)

train_count = extract_patches(
    TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, TRAIN_PATCH_IMG, TRAIN_PATCH_MASK,
    PATCH_SIZE, STRIDE, MIN_FOREGROUND, split_name="TRAIN"
)
test_count = extract_patches(
    TEST_IMAGE_DIR, TEST_MASK_DIR, TEST_PATCH_IMG, TEST_PATCH_MASK,
    PATCH_SIZE, STRIDE, MIN_FOREGROUND, split_name="TEST"
)
print(f"Final patch counts -> Train: {train_count} | Test: {test_count}")


# ============================================================
# CELL 4 — Dataset, transforms, dataloaders
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Transpose(p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])


class MoNuSegPatchDataset(Dataset):
    """
    Loads pre-extracted 256x256 patches from disk.
    Images: RGB .png (cv2 loads as BGR -> converted here)
    Masks : grayscale .png with values 0 or 255
    Returns: image tensor [3,256,256], mask tensor [1,256,256] float
    """
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir  = mask_dir
        self.transform = transform
        self.image_files = sorted(os.listdir(image_dir))
        self.mask_files  = sorted(os.listdir(mask_dir))
        assert len(self.image_files) == len(self.mask_files), \
            f"Mismatch: {len(self.image_files)} images vs {len(self.mask_files)} masks"
        print(f"Dataset loaded: {len(self.image_files)} patches from {image_dir}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])
        mask = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        return image, mask.unsqueeze(0)


# ─── Build datasets (80/20 train/val split) ───
full_train_dataset = MoNuSegPatchDataset(TRAIN_PATCH_IMG, TRAIN_PATCH_MASK, get_train_transforms())
test_dataset = MoNuSegPatchDataset(TEST_PATCH_IMG, TEST_PATCH_MASK, get_val_transforms())

val_size   = int(0.2 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
val_dataset.dataset.transform = get_val_transforms()  # no augmentation on val

print(f"Split -> Train: {train_size} | Val: {val_size} | Test: {len(test_dataset)}")

# ─── Dataloaders ───
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

# ─── Sanity check ───
images, masks = next(iter(train_loader))
print(f"Batch check -> images: {images.shape} | masks: {masks.shape}")
print(f"Image range: [{images.min():.2f}, {images.max():.2f}]")
print(f"Mask unique values: {masks.unique()}")

Please upload your kaggle.json file


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/tuanledinh/monuseg2018
License(s): unknown
100% 79.1M/79.1M [00:05<00:00, 15.6MB/s]

Dataset extracted successfully!
Mask filenames cleaned: /content/MoNuSeg_Original/kmms_training/kmms_training/masks
Mask filenames cleaned: /content/MoNuSeg_Original/kmms_test/kmms_test/masks


Patching TRAIN: 100%|██████████| 24/24 [00:04<00:00,  5.62it/s]


TRAIN: 864 patches saved | 0 background patches skipped


Patching TEST: 100%|██████████| 58/58 [00:01<00:00, 33.11it/s]

TEST: 331 patches saved | 7 background patches skipped
Final patch counts -> Train: 864 | Test: 331
Dataset loaded: 864 patches from /content/MoNuSeg_Patches/train/images
Dataset loaded: 331 patches from /content/MoNuSeg_Patches/test/images
Split -> Train: 692 | Val: 172 | Test: 331


Batch check -> images: torch.Size([4, 3, 256, 256]) | masks: torch.Size([4, 1, 256, 256])
Image range: [-2.04, 2.64]
Mask unique values: tensor([0., 1.])


In [3]:
TRAIN_IMAGE_DIR = "/content/MoNuSeg_Original/kmms_training/kmms_training/images"
TRAIN_MASK_DIR  = "/content/MoNuSeg_Original/kmms_training/kmms_training/masks"
TEST_IMAGE_DIR  = "/content/MoNuSeg_Original/kmms_test/kmms_test/images"
TEST_MASK_DIR   = "/content/MoNuSeg_Original/kmms_test/kmms_test/masks"

TRAIN_PATCH_IMG  = "/content/MoNuSeg_Patches/train/images"
TRAIN_PATCH_MASK = "/content/MoNuSeg_Patches/train/masks"
TEST_PATCH_IMG   = "/content/MoNuSeg_Patches/test/images"
TEST_PATCH_MASK  = "/content/MoNuSeg_Patches/test/masks"

PATCH_SIZE     = 256
STRIDE         = 128
MIN_FOREGROUND = 0.02

print("✓ Paths set")

✓ Paths set


In [4]:
import shutil

def fix_mask_filenames(mask_dir):
    for f in os.listdir(mask_dir):
        old = os.path.join(mask_dir, f)
        new = os.path.join(mask_dir, f.replace(" ", ""))
        if old != new:
            os.rename(old, new)
    print(f"✓ Mask filenames cleaned: {mask_dir}")


def extract_patches(image_dir, mask_dir, out_img_dir, out_mask_dir,
                    patch_size, stride, min_fg, split_name):
    for d in [out_img_dir, out_mask_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d)

    count, skipped = 0, 0

    for img_name in tqdm(sorted(os.listdir(image_dir)), desc=f"Patching {split_name}"):
        img_path  = os.path.join(image_dir, img_name)
        mask_name = img_name.replace(".tif", ".png").replace(" ", "")
        mask_path = os.path.join(mask_dir, mask_name)

        image = cv2.imread(img_path)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            print(f"  ✗ Missing image: {img_path}"); continue
        if mask is None:
            print(f"  ✗ Missing mask : {mask_path}"); continue

        H, W = image.shape[:2]
        if mask.shape[:2] != (H, W):
            mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

        mask = (mask > 127).astype(np.uint8) * 255

        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):
                img_patch  = image[y:y+patch_size, x:x+patch_size]
                mask_patch = mask[y:y+patch_size,  x:x+patch_size]

                if img_patch.shape  != (patch_size, patch_size, 3): continue
                if mask_patch.shape != (patch_size, patch_size):    continue

                fg_ratio = np.sum(mask_patch > 0) / (patch_size * patch_size)
                if fg_ratio < min_fg:
                    skipped += 1
                    continue

                cv2.imwrite(os.path.join(out_img_dir,  f"{count:05d}.png"), img_patch)
                cv2.imwrite(os.path.join(out_mask_dir, f"{count:05d}.png"), mask_patch)
                count += 1

    print(f"✓ {split_name}: {count} saved | {skipped} skipped")
    return count


fix_mask_filenames(TRAIN_MASK_DIR)
fix_mask_filenames(TEST_MASK_DIR)

train_count = extract_patches(
    TRAIN_IMAGE_DIR, TRAIN_MASK_DIR,
    TRAIN_PATCH_IMG, TRAIN_PATCH_MASK,
    PATCH_SIZE, STRIDE, MIN_FOREGROUND, "TRAIN"
)
test_count = extract_patches(
    TEST_IMAGE_DIR, TEST_MASK_DIR,
    TEST_PATCH_IMG, TEST_PATCH_MASK,
    PATCH_SIZE, STRIDE, MIN_FOREGROUND, "TEST"
)
print(f"Final counts → Train: {train_count} | Test: {test_count}")

✓ Mask filenames cleaned: /content/MoNuSeg_Original/kmms_training/kmms_training/masks
✓ Mask filenames cleaned: /content/MoNuSeg_Original/kmms_test/kmms_test/masks


Patching TRAIN: 100%|██████████| 24/24 [00:04<00:00,  5.73it/s]


✓ TRAIN: 864 saved | 0 skipped


Patching TEST: 100%|██████████| 58/58 [00:01<00:00, 30.40it/s]

✓ TEST: 331 saved | 7 skipped
Final counts → Train: 864 | Test: 331


In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Transpose(p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])


class MoNuSegPatchDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir   = image_dir
        self.mask_dir    = mask_dir
        self.transform   = transform
        self.image_files = sorted(os.listdir(image_dir))
        self.mask_files  = sorted(os.listdir(mask_dir))
        assert len(self.image_files) == len(self.mask_files), \
            f"Mismatch: {len(self.image_files)} images vs {len(self.mask_files)} masks"
        print(f"✓ {len(self.image_files)} patches from {image_dir}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image = cv2.cvtColor(
            cv2.imread(os.path.join(self.image_dir, self.image_files[idx])),
            cv2.COLOR_BGR2RGB
        )
        mask = (cv2.imread(
            os.path.join(self.mask_dir, self.mask_files[idx]),
            cv2.IMREAD_GRAYSCALE
        ) > 127).astype(np.float32)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"]

        return image, mask.unsqueeze(0)


n_total   = len(os.listdir(TRAIN_PATCH_IMG))
all_idx   = np.arange(n_total)
train_idx, val_idx = train_test_split(
    all_idx, test_size=0.2, random_state=42, shuffle=True
)

train_dataset = Subset(
    MoNuSegPatchDataset(TRAIN_PATCH_IMG, TRAIN_PATCH_MASK, get_train_transforms()),
    train_idx
)
val_dataset = Subset(
    MoNuSegPatchDataset(TRAIN_PATCH_IMG, TRAIN_PATCH_MASK, get_val_transforms()),
    val_idx
)
test_dataset = MoNuSegPatchDataset(
    TEST_PATCH_IMG, TEST_PATCH_MASK, get_val_transforms()
)

print(f"Split → Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=2, shuffle=False,
                          num_workers=2, pin_memory=True)

imgs, msks = next(iter(train_loader))
print(f"Batch → images: {imgs.shape} | masks: {msks.shape}")
print(f"Mask unique values: {msks.unique()}")
print("✓ DataLoaders ready")

✓ 864 patches from /content/MoNuSeg_Patches/train/images
✓ 864 patches from /content/MoNuSeg_Patches/train/images
✓ 331 patches from /content/MoNuSeg_Patches/test/images
Split → Train: 691 | Val: 173 | Test: 331
Batch → images: torch.Size([2, 3, 256, 256]) | masks: torch.Size([2, 1, 256, 256])
Mask unique values: tensor([0., 1.])
✓ DataLoaders ready


In [6]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, max(channels//reduction, 4), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels//reduction, 4), channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(
            self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x))
        )


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg  = torch.mean(x, dim=1, keepdim=True)
        mx,_ = torch.max(x,  dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))


class SkipTransformer(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn  = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads,
            batch_first=True, dropout=0.0
        )
        self.norm2 = nn.LayerNorm(channels)
        self.mlp   = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels)
        )
    def forward(self, x):
        B, C, H, W = x.shape
        xf = x.flatten(2).transpose(1, 2)
        n  = self.norm1(xf)
        xf = xf + self.attn(n, n, n)[0]
        xf = xf + self.mlp(self.norm2(xf))
        return xf.transpose(1, 2).reshape(B, C, H, W)


class BottleneckTransformer(nn.Module):
    def __init__(self, channels=1024, num_heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn  = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads,
            batch_first=True, dropout=0.0
        )
        self.norm2 = nn.LayerNorm(channels)
        self.mlp   = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels)
        )
    def forward(self, x):
        B, C, H, W = x.shape
        xf = x.flatten(2).transpose(1, 2)
        n  = self.norm1(xf)
        xf = xf + self.attn(n, n, n)[0]
        xf = xf + self.mlp(self.norm2(xf))
        return xf.transpose(1, 2).reshape(B, C, H, W)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.cbam = CBAM(skip_ch)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:],
                              mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, self.cbam(skip)], dim=1))


class BoundaryRefinement(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )
    def forward(self, pred, uncertainty):
        return self.net(torch.cat([pred, uncertainty], dim=1))


class TransformerAugmentedUNet(nn.Module):
    def __init__(self, in_channels=3, noise_std=0.1):
        super().__init__()
        self.noise_std = noise_std

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64,  128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        # Skip transformer on s4 only
        self.trans4 = SkipTransformer(channels=512, num_heads=4)

        # Bottleneck
        self.bottleneck             = DoubleConv(512, 1024)
        self.bottleneck_transformer = BottleneckTransformer(1024, num_heads=8)

        # Main decoder
        self.d1 = DecoderBlock(1024, 512, 512)
        self.d2 = DecoderBlock(512,  256, 256)
        self.d3 = DecoderBlock(256,  128, 128)
        self.d4 = DecoderBlock(128,   64,  64)
        self.main_head = nn.Conv2d(64, 1, 1)

        # Auxiliary decoder
        self.a1 = DecoderBlock(1024, 512, 512)
        self.a2 = DecoderBlock(512,  256, 256)
        self.a3 = DecoderBlock(256,  128, 128)
        self.a4 = DecoderBlock(128,   64,  64)
        self.aux_head = nn.Conv2d(64, 1, 1)

        # Boundary refinement
        self.refine = BoundaryRefinement()

    def forward(self, x):
        # Encoder (gradient checkpointing saves ~40% memory)
        s1 = checkpoint(self.enc1, x,   use_reentrant=False); p1 = self.pool(s1)
        s2 = checkpoint(self.enc2, p1,  use_reentrant=False); p2 = self.pool(s2)
        s3 = checkpoint(self.enc3, p2,  use_reentrant=False); p3 = self.pool(s3)
        s4 = checkpoint(self.enc4, p3,  use_reentrant=False); p4 = self.pool(s4)

        s4 = checkpoint(self.trans4, s4, use_reentrant=False)

        b  = checkpoint(self.bottleneck, p4,              use_reentrant=False)
        b  = checkpoint(self.bottleneck_transformer, b,   use_reentrant=False)

        # Main decoder
        d  = checkpoint(self.d1, b,  s4, use_reentrant=False)
        d  = checkpoint(self.d2, d,  s3, use_reentrant=False)
        d  = checkpoint(self.d3, d,  s2, use_reentrant=False)
        d  = checkpoint(self.d4, d,  s1, use_reentrant=False)
        main_logits = self.main_head(d)

        # Auxiliary decoder (noise only during training)
        noise = self.noise_std * torch.randn_like(b) \
                if self.training else torch.zeros_like(b)
        a = checkpoint(self.a1, b+noise, s4, use_reentrant=False)
        a = checkpoint(self.a2, a,       s3, use_reentrant=False)
        a = checkpoint(self.a3, a,       s2, use_reentrant=False)
        a = checkpoint(self.a4, a,       s1, use_reentrant=False)
        aux_logits = self.aux_head(a)

        # Uncertainty + refinement
        main_prob   = torch.sigmoid(main_logits)
        aux_prob    = torch.sigmoid(aux_logits)
        uncertainty = torch.abs(main_prob - aux_prob)
        residual    = self.refine(main_prob, uncertainty)
        refined     = torch.sigmoid(main_logits + residual)

        return main_logits, aux_logits, refined, uncertainty


# Memory check
model = TransformerAugmentedUNet().to(device)
with torch.no_grad():
    with autocast():
        out = model(torch.randn(2, 3, 256, 256).to(device))
for i, o in enumerate(out):
    print(f"Output {i}: {o.shape}")

free = torch.cuda.mem_get_info()[0] / 1e9
print(f"\nGPU free after model load: {free:.2f} GB")
print("✓ Safe to train" if free > 1.5 else "⚠ Very tight — reduce batch to 1")

Output 0: torch.Size([2, 1, 256, 256])
Output 1: torch.Size([2, 1, 256, 256])
Output 2: torch.Size([2, 1, 256, 256])
Output 3: torch.Size([2, 1, 256, 256])

GPU free after model load: 14.93 GB
✓ Safe to train


In [7]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        inter   = (probs * targets).sum()
        return 1 - (2.*inter + self.smooth) / (
            probs.sum() + targets.sum() + self.smooth)

class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return 0.5*self.bce(logits, targets) + 0.5*self.dice(logits, targets)

class RefinedLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, probs, targets):
        p, t = probs.float().view(-1), targets.float().view(-1)
        dice = 1 - (2*(p*t).sum() + self.smooth) / (
            p.sum() + t.sum() + self.smooth)
        return dice  # pure dice, avoids autocast BCE conflict

criterion_main    = DiceBCELoss()
criterion_aux     = DiceBCELoss()
criterion_refined = RefinedLoss()

def compute_total_loss(main_logits, aux_logits, refined, masks):
    loss_main    = criterion_main(main_logits, masks)
    loss_aux     = criterion_aux(aux_logits,   masks)
    loss_refined = criterion_refined(refined,  masks)
    return 0.5*loss_main + 0.2*loss_aux + 0.3*loss_refined

print("✓ Loss functions ready")

✓ Loss functions ready


In [8]:
scaler = GradScaler()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(loader, desc="Train"):
        images = images.to(device)
        masks  = masks.to(device).float()
        optimizer.zero_grad()

        with autocast():
            main_logits, aux_logits, refined, _ = model(images)
            loss_main = criterion_main(main_logits, masks)
            loss_aux  = criterion_aux(aux_logits,   masks)

        # Refined loss outside autocast — avoids BCE + float16 conflict
        loss_refined = criterion_refined(refined.float(), masks.float())
        loss = 0.5*loss_main + 0.2*loss_aux + 0.3*loss_refined

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    return running_loss / len(loader)


def validate(model, loader, device, threshold=0.5):
    model.eval()
    total_loss = total_dice = total_iou = total_pa = 0.0

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Val"):
            images = images.to(device)
            masks  = masks.to(device).float()

            with autocast():
                main_logits, aux_logits, refined, _ = model(images)
                loss_main = criterion_main(main_logits, masks)
                loss_aux  = criterion_aux(aux_logits,   masks)

            loss_refined = criterion_refined(refined.float(), masks.float())
            loss = 0.5*loss_main + 0.2*loss_aux + 0.3*loss_refined
            total_loss += loss.item()

            preds = (refined.float() > threshold).float()
            inter = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))

            total_dice += ((2*inter+1e-7)/(union+1e-7)).mean().item()
            total_iou  += ((inter+1e-7)/(union-inter+1e-7)).mean().item()
            total_pa   += (preds==masks).float().mean(dim=(1,2,3)).mean().item()

    n = len(loader)
    return total_loss/n, total_dice/n, total_iou/n, total_pa/n

print("✓ Train/Val functions ready")

✓ Train/Val functions ready


In [10]:
model = TransformerAugmentedUNet(in_channels=3, noise_std=0.1).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-5
)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / (120 - warmup)
    return 0.5 * (1 + torch.cos(torch.tensor(3.14159 * progress)).item())

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_dice = 0.0
patience  = 20
counter   = 0
scaler    = GradScaler()

for epoch in range(120):

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_dice, val_iou, val_pa = validate(
        model, val_loader, device, threshold=0.5
    )
    scheduler.step()

    lr_now = optimizer.param_groups[0]['lr']
    print(
        f"Epoch {epoch+1:03d} | "
        f"LR: {lr_now:.6f} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f} | "
        f"Dice: {val_dice:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"PA: {val_pa:.4f}"
    )

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "best_monuseg.pth")
        print(f"  ✓ Saved — Dice: {best_dice:.4f}")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping.")
            break

Train:   3%|▎         | 10/345 [00:03<01:43,  3.23it/s]


KeyboardInterrupt: 

In [11]:
model.load_state_dict(torch.load("best_monuseg.pth"))
print("✓ Best model loaded")

test_loss, test_dice, test_iou, test_pa = validate(
    model, test_loader, device, threshold=0.5
)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Pixel Accuracy : {test_pa:.4f}")
print(f"Dice Score     : {test_dice:.4f}")
print(f"IoU Score      : {test_iou:.4f}")
print("="*50)

✓ Best model loaded


Val: 100%|██████████| 166/166 [00:12<00:00, 13.47it/s]


TEST SET RESULTS
Pixel Accuracy : 0.8724
Dice Score     : 0.6947
IoU Score      : 0.5607


In [18]:
import matplotlib.pyplot as plt

model.eval()

all_images = []
all_masks = []
all_refined = []
all_uncertainty = []

num_batches_to_display = 3 # Number of batches to display

# Create a fresh iterator for the validation loader to get new batches each time the cell is run
val_iter = iter(val_loader)

for i in range(num_batches_to_display):
    try:
        images_batch, masks_batch = next(val_iter)
        images_batch = images_batch.to(device)

        with torch.no_grad():
            with autocast():
                _, _, refined_batch, uncertainty_batch = model(images_batch)

        all_images.append(images_batch.cpu().numpy())
        all_masks.append(masks_batch.cpu().numpy())
        all_refined.append(refined_batch.float().cpu().numpy())
        all_uncertainty.append(uncertainty_batch.float().cpu().numpy())
    except StopIteration:
        print(f"Reached end of validation loader after {i} batches. Displaying available images.")
        break

if not all_images:
    print("No images to display. Validation loader might be exhausted or empty.")
else:
    # Concatenate all batches to create a single set of arrays for plotting
    images_np   = np.concatenate(all_images, axis=0)
    masks_np    = np.concatenate(all_masks, axis=0)
    refined     = np.concatenate(all_refined, axis=0)
    uncertainty = np.concatenate(all_uncertainty, axis=0)

    # Dynamically set the number of rows based on the total number of images
    fig, axes = plt.subplots(images_np.shape[0], 4, figsize=(16, images_np.shape[0] * 4)) # Adjust figure size dynamically
    titles = ["Input", "Ground truth", "Prediction", "Uncertainty"]

    for i in range(images_np.shape[0]):
        img = images_np[i].transpose(1,2,0)
        img = img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
        img = np.clip(img, 0, 1)

        pred = (refined[i,0] > 0.5).astype(np.float32)
        unc  = uncertainty[i,0]

        # Handle case where there's only one image in total, so `axes` is 1D
        current_axes = axes[i] if images_np.shape[0] > 1 else axes

        current_axes[0].imshow(img);                         current_axes[0].set_title(titles[0])
        current_axes[1].imshow(masks_np[i,0], cmap='gray');  current_axes[1].set_title(titles[1])
        current_axes[2].imshow(pred, cmap='gray');            current_axes[2].set_title(titles[2])
        current_axes[3].imshow(unc,  cmap='hot');             current_axes[3].set_title(titles[3])

        for ax in current_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.savefig("predictions_val_multiple_batches.png", dpi=150, bbox_inches='tight') # Changed filename
    plt.show()
    print("✓ Saved predictions_val_multiple_batches.png")

Output hidden; open in https://colab.research.google.com to view.